# P18 review round 3 controls
R3 / R3-p: 3 random lessons from the P13 G3 / G3-p guide's candidates (no judgement). G3s: the guide's task text without the feature sentence only; G3s-pl: placebo, an unrelated sentence removed (TF03, TF05). G3d-tr / G3d-tr-pc: the data profile without test-set numbers. G3h / G3h-pc: the guide sees the suite's first hint-level wording (guide only). X3 / X3-pc: embedding top-20 re-scored by a cross-encoder, top 3. Predictions: `quick_runs/predictions/P18.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T14, "G3n")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T14, "G3n-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3n                 5                  2          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3n-p,20,20,10,9,5,0,0,4


In [5]:
T16 = [json.loads(l) for l in open(REPO / "quick_runs/results/P16/guide_trace.jsonl")]
T18 = [json.loads(l) for l in open(REPO / "quick_runs/results/P18/guide_trace.jsonl")]
for t in T13 + T16 + T18:  # arms without a guide: seen = accepted = shown
    t.setdefault("seen", t["shown"]); t.setdefault("accepted", t["shown"])
print(Counter(t["arm"] for t in T18))
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T16, "G3t"), stage1(T16, "G3d"), stage1(T18, "R3"), stage1(T18, "G3s"),
                   stage1(T18, "G3s-pl"), stage1(T18, "G3d-tr"), stage1(T18, "G3h"), stage1(T13, "E"), stage1(T18, "X3")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "E-p"), wrong(T13, "G3-p"), wrong(T18, "R3-p"), wrong(T16, "E-pc"), wrong(T16, "G3-pc"),
                   wrong(T16, "G3d-pc"), wrong(T18, "G3d-tr-pc"), wrong(T18, "G3h-pc"), wrong(T18, "X3-pc")]).set_index("arm")
print(s1.to_string()); wr

Counter({'G3d-tr': 30, 'R3': 30, 'X3': 30, 'G3d-tr-pc': 20, 'R3-p': 20, 'G3h': 20, 'G3h-pc': 20, 'X3-pc': 20, 'G3s': 10, 'G3s-pl': 10})
        sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                      
G3                     5                  0          5                 5                  1          5                  4
G3t                    5                  5          5                 5                  5          5                  5
G3d                    5                  0          5                 5                  5          5                  0
R3                     5                  5          5                 4                  4          5                  2
G3s                    5                  5          5                 5                  5          5                  3
G3s-pl    

,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
E-p,20,10,10,10,5,0,0,5
G3-p,20,20,5,5,4,0,0,1
R3-p,20,10,10,10,0,2,3,5
E-pc,20,20,20,20,5,5,5,5
G3-pc,20,20,20,20,5,5,5,5
G3d-pc,20,20,8,8,0,0,3,5
G3d-tr-pc,20,20,15,15,0,5,5,5
G3h-pc,20,20,5,5,5,0,0,0
X3-pc,20,15,15,15,5,5,5,0


In [6]:
# Reasons for rejecting fix lessons and accepting/rejecting wrong lessons (pre-set regex + contradict-only rule)
NOC = r"task (says|asks|states|requires|instructs)|instruction|use all|all available"
TASKWORD = r"prompt|task|requirement|constraint"
def cites_instr(reason):
    if not re.search(INSTR, reason, re.I): return False
    return bool(re.search(NOC, reason, re.I) or re.search(TASKWORD, reason, re.I))
rows = []
for t in T18:
    for r in t.get("rounds", []):
        for j in r.get("judgements") or []:
            try: i = int(j.get("id"))
            except (TypeError, ValueError): continue
            kind = "fix" if i in FIX.get(t["task"], set()) else "wrong" if i in WRONG_IDS else "contra" if (i in CONTRA3 and t["task"] == "sf_tf03_correlation_causation") else None
            if kind is None or i not in r["candidates"]: continue
            reason = str(j.get("reason", ""))
            rows.append({"arm": t["arm"], "task": t["task"][:7], "run": t["run"], "id": i, "kind": kind,
                         "useful": j.get("useful") is True, "instr": cites_instr(reason), "reason": reason})
jr = pd.DataFrame(rows)
rej = jr[~jr.useful]
print(rej.groupby(["arm", "kind"]).instr.agg(["sum", "count", "mean"]).round(2).to_string())
for _, r in jr.groupby(["arm", "kind", "task", "useful"]).head(1).iterrows(): print(f"{r.arm} {r.kind} {r.task} useful={r.useful} [{r.id}]: {r.reason[:240]}")

                  sum  count  mean
arm       kind                    
G3d-tr    contra    0      1  0.00
          fix      40     50  0.80
G3d-tr-pc contra    1      3  0.33
          fix      15     29  0.52
          wrong     3     32  0.09
G3h       contra    6     10  0.60
          fix       0     38  0.00
G3h-pc    contra    5      9  0.56
          fix       0     20  0.00
          wrong    39     48  0.81
G3s       contra    1      7  0.14
          fix       3     36  0.08
G3s-pl    contra    0      1  0.00
          fix      38     48  0.79
G3d-tr contra sf_tf03 useful=True [130]: The data profile shows 'column_x' has a 1.0 correlation with the label. This note warns against accidentally dropping such columns during cleaning, which is critical here given the prompt's specific instruction to 'keep only numerical featu
G3d-tr fix sf_tf03 useful=False [135]: This note suggests applying causal reasoning to exclude features. However, the task explicitly mandates keeping 'only n

In [7]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P12_C1", "P13_E", "P13_G3", "P16_G3t", "P16_G3d", "P18_R3", "P18_G3s", "P18_G3spl", "P18_G3dtr", "P18_X3", "P13_Ep", "P13_G3p", "P18_R3p", "P16_Epc", "P16_G3pc", "P16_G3dpc", "P18_G3dtrpc", "P18_X3pc"]
NAME = dict(zip(ARMS, ["A0 none", "C1 own", "E emb", "G3", "G3t", "G3d", "R3", "G3s", "G3s-pl", "G3d-tr", "X3", "E-p", "G3-p", "R3-p", "E-pc", "G3-pc", "G3d-pc", "G3d-tr-pc", "X3-pc"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G3': (30, {'final_answer': 29, 'max_steps': 1}), 'P16_G3t': (30, {'final_answer': 28, 'max_steps': 2}), 'P16_G3d': (30, {'final_answer': 28, 'max_steps': 2}), 'P18_R3': (30, {'final_answer': 29, 'max_steps': 1}), 'P18_G3s': (10, {'final_answer': 10}), 'P18_G3spl': (10, {'final_answer': 10}), 'P18_G3dtr': (30, {'final_answer': 30}), 'P18_X3': (30, {'final_answer': 30}), 'P13_Ep': (20, {'final_answer': 19, 'max_steps': 1}), 'P13_G3p': (20, {'final_answer': 20}), 'P18_R3p': (20, {'final_answer': 20}), 'P16_Epc': (20, {'final_answer': 20}), 'P16_G3pc': (20, {'final_answer': 20}), 'P16_G3dpc': (20, {'final_answer': 20}), 'P18_G3dtrpc': (20, {'final_answer': 20}), 'P18_X3pc': (20, {'final_answer': 20})}
pooled: {'A0 none': 0.075, 'C1 own': 0.75, 'E emb': 0.475, 'G3': 0.5, 'G3t': 0.425, 'G3d': 0.6, 'R3': 0.425, 'G3s': 0.075, 'G3s-pl': 0.0, 'G3d

,task,check,A0 none,C1 own,E emb,G3,G3t,G3d,R3,G3s,G3s-pl,G3d-tr,X3,E-p,G3-p,R3-p,E-pc,G3-pc,G3d-pc,G3d-tr-pc,X3-pc
0,sf_tf01,checks_class_distribution,0,5,5,4,4,2,0,0,0,1,4,3,3,3,1,0,2,3,0
1,sf_tf01,uses_class_weight,0,5,5,5,5,5,4,0,0,5,4,5,5,4,0,0,5,5,0
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,5,5,5,5,4,0,0,5,5,5,5,4,0,0,5,5,5
4,sf_tf02,handles_unit_suffix_in_height,0,4,4,4,3,3,4,0,0,5,0,4,5,4,0,0,5,0,0
5,sf_tf03,column_x_included,0,5,0,1,0,5,3,2,0,0,5,0,0,3,0,0,1,0,0
6,sf_tf03,applies_causal_reasoning,0,5,0,1,0,4,2,1,0,0,5,0,0,4,0,0,1,0,0
7,sf_tf05,used_leakage_features,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
# TF05 leaking columns kept (of 4) per run
def leak_kept(a, r):
    j = json.load(open(RUNS / a / r["task_id"] / r["run"] / "judge.json"))
    return len(Counter(tuple(sorted(d.get("leakage_features_used") or [])) for d in j["draws"]).most_common(1)[0][0])
tf05_kept = {NAME[a]: [leak_kept(a, r) for r in sorted(recs[a], key=lambda r: r["run"]) if r["task_id"] == "sf_tf05_credit_card_leakage"] for a in ARMS}
tf05_kept

{'A0 none': [4, 4, 4, 4, 4],
 'C1 own': [1, 0, 1, 1, 1],
 'E emb': [4, 2, 2, 4, 4],
 'G3': [4, 4, 4, 4, 4],
 'G3t': [4, 4, 4, 4, 4],
 'G3d': [4, 4, 4, 4, 4],
 'R3': [4, 2, 2, 3, 4],
 'G3s': [4, 4, 4, 4, 4],
 'G3s-pl': [4, 4, 4, 4, 4],
 'G3d-tr': [4, 4, 4, 4, 4],
 'X3': [4, 4, 4, 4, 4],
 'E-p': [4, 4, 4, 4, 4],
 'G3-p': [4, 4, 4, 4, 4],
 'R3-p': [4, 4, 2, 4, 2],
 'E-pc': [4, 4, 4, 4, 4],
 'G3-pc': [4, 4, 4, 4, 4],
 'G3d-pc': [4, 4, 4, 4, 4],
 'G3d-tr-pc': [4, 4, 4, 4, 4],
 'X3-pc': [4, 4, 4, 4, 4]}

In [9]:
# TF03 and TF05 runs (of 5) for the two-task arms: checks passed, and runs that left column_x out / kept no leaking column
tf35 = targeted[targeted.task.isin(["sf_tf03", "sf_tf05"])].set_index(["task", "check"])[["A0 none", "C1 own", "G3", "G3t", "G3d", "G3s", "G3s-pl", "G3d-tr"]]
tf35

A0 none  C1 own  G3  G3t  G3d  G3s  G3s-pl  G3d-tr
task    check                                                                       
sf_tf03 column_x_included               0       5   1    0    5    2       0       0
        applies_causal_reasoning        0       5   1    0    4    1       0       0
sf_tf05 used_leakage_features           0       1   0    0    0    0       0       0

In [10]:
# Trust funnel: wrong lesson shown -> followed. -p: named column left out (executed variables). -pc: targeted check failed.
TR = {(t["arm"], t["task"], t["run"]): t for t in T13 + T16 + T18}
def uses(r, col):
    v = r["executed"].get("variables") or []
    return any(col == x or x.startswith(col + "_") for x in v)
FOLLOW_C = {"sf_tf01_goal_imbalance": ("uses_class_weight", False), "sf_tf02_player_rep_norm": ("handles_unit_suffix_in_height", False),
            "sf_tf03_correlation_causation": ("column_x_included", False), "sf_tf05_credit_card_leakage": ("used_leakage_features", False)}
GARM = {"P13_Ep": "E-p", "P13_G3p": "G3-p", "P18_R3p": "R3-p", "P16_G3dpc": "G3d-pc", "P18_G3dtrpc": "G3d-tr-pc", "P18_X3pc": "X3-pc", "P16_Epc": "E-pc"}
rows = []
for a, g in GARM.items():
    for r in recs[a]:
        t = TR.get((g, r["task_id"], r["run"]))
        if t is None: continue
        shown = bool(set(t["shown"]) & WRONG_IDS)
        if g.endswith("-pc"):
            key, val = FOLLOW_C[r["task_id"]]; followed = r["checks"].get(key) is val
        else:
            followed = not uses(r, WRONG_COL[r["task_id"]])
        rows.append({"arm": g, "task": r["task_id"][:7], "run": r["run"], "shown": shown, "followed": followed})
fun = pd.DataFrame(rows)
print(fun.groupby(["arm", "shown"]).followed.agg(["sum", "count"]).to_string())
fun.groupby(["arm", "task"])[["shown", "followed"]].sum().unstack()

                 sum  count
arm       shown            
E-p       False    0     10
          True    10     10
E-pc      True    20     20
G3-p      False    0     15
          True     5      5
G3d-pc    False    1     12
          True     8      8
G3d-tr-pc False    0      5
          True    15     15
R3-p      False    0     10
          True     9     10
X3-pc     False    5      5
          True    15     15


shown                         followed                        
task      sf_tf01 sf_tf02 sf_tf03 sf_tf05  sf_tf01 sf_tf02 sf_tf03 sf_tf05
arm                                                                       
E-p             5       0       0       5        5       0       0       5
E-pc            5       5       5       5        5       5       5       5
G3-p            4       0       0       1        4       0       0       1
G3d-pc          0       0       3       5        0       0       4       5
G3d-tr-pc       0       5       5       5        0       5       5       5
R3-p            0       2       3       5        0       1       3       5
X3-pc           5       5       5       0        5       5       5       5

In [11]:
# Harm, cost and exact tests (two-sided Fisher; runs or check-runs as the unit)
from scipy.stats import fisher_exact
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
gcost = {}
for arm in ("G3s", "G3s-pl", "G3d-tr", "G3h", "G3h-pc"):
    gcost[arm] = statistics.mean(sum(x["usage"]["prompt_tokens"] + x["usage"]["completion_tokens"] for x in t["rounds"] if x.get("usage")) for t in T18 if t["arm"] == arm)
passed = (pooled * 40).round().astype(int)
def fp(a, b, n): return round(fisher_exact([[a, n - a], [b, n - b]])[1], 4)
fix35 = lambda arm: int(s1.loc[arm, ["sf_tf03 fix shown", "sf_tf05 fix shown"]].sum())
fisher = {"fix shown TF03+TF05 G3 vs G3s (of 10)": fp(fix35("G3"), fix35("G3s"), 10),
          "fix shown TF03+TF05 G3s vs G3s-pl (of 10)": fp(fix35("G3s"), fix35("G3s-pl"), 10),
          "fix shown TF03+TF05 G3 vs G3h (of 10)": fp(fix35("G3"), fix35("G3h"), 10),
          "fix shown TF03+TF05 G3 vs R3 (of 10)": fp(fix35("G3"), fix35("R3"), 10),
          "wrong shown G3-p vs R3-p": fp(int(wr.loc["G3-p", "wrong shown"]), int(wr.loc["R3-p", "wrong shown"]), 20),
          "wrong shown G3-pc vs G3h-pc": fp(int(wr.loc["G3-pc", "wrong shown"]), int(wr.loc["G3h-pc", "wrong shown"]), 20),
          "wrong shown G3d-pc vs G3d-tr-pc": fp(int(wr.loc["G3d-pc", "wrong shown"]), int(wr.loc["G3d-tr-pc", "wrong shown"]), 20),
          "passed G3 vs R3": fp(passed["G3"], passed["R3"], 40), "passed E vs X3": fp(passed["E emb"], passed["X3"], 40),
          "passed G3d vs G3d-tr": fp(passed["G3d"], passed["G3d-tr"], 40), "passed G3-pc vs G3d-tr-pc": fp(passed["G3-pc"], passed["G3d-tr-pc"], 40)}
print("guide tokens per run:", {k: round(v) for k, v in gcost.items()}); print(fisher); scores

guide tokens per run: {'G3s': 10573, 'G3s-pl': 9235, 'G3d-tr': 8213, 'G3h': 7402, 'G3h-pc': 6237}
{'fix shown TF03+TF05 G3 vs G3s (of 10)': np.float64(0.0001), 'fix shown TF03+TF05 G3s vs G3s-pl (of 10)': np.float64(0.0007), 'fix shown TF03+TF05 G3 vs G3h (of 10)': np.float64(0.1409), 'fix shown TF03+TF05 G3 vs R3 (of 10)': np.float64(0.0011), 'wrong shown G3-p vs R3-p': np.float64(0.1908), 'wrong shown G3-pc vs G3h-pc': np.float64(0.0), 'wrong shown G3d-pc vs G3d-tr-pc': np.float64(0.0536), 'passed G3 vs R3': np.float64(0.6541), 'passed E vs X3': np.float64(0.5021), 'passed G3d vs G3d-tr': np.float64(0.117), 'passed G3-pc vs G3d-tr-pc': np.float64(0.0001)}


arm,A0 none,C1 own,E emb,E-p,E-pc,G3,G3-p,G3-pc,G3d,G3d-pc,G3d-tr,G3d-tr-pc,G3s,G3s-pl,G3t,R3,R3-p,X3,X3-pc
task,,,,,,,,,,,,,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.765000e-01,7.713000e-01,7.387000e-01,7.600000e-01,7.715000e-01,7.449000e-01,7.865000e-01,7.658000e-01,7.659000e-01,7.657000e-01,7.767000e-01,NaN,NaN,7.713000e-01,7.764000e-01,7.827000e-01,7.878000e-01,7.931000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.650909e+06,4.650909e+06,4.657555e+06,4.664201e+06,4.650909e+06,4.650909e+06,4.686842e+06,4.655340e+06,4.650909e+06,4.650909e+06,4.664201e+06,NaN,NaN,4.650909e+06,4.655340e+06,4.651365e+06,4.664201e+06,4.664201e+06
sf_tf03_correlation_causation,2.769462e+07,4.755266e+06,2.769464e+07,2.769464e+07,2.769463e+07,2.310314e+07,2.769458e+07,2.769463e+07,4.759155e+06,2.311203e+07,2.769459e+07,2.769461e+07,1.852822e+07,2.769459e+07,2.769464e+07,1.624767e+07,1.623956e+07,4.768432e+06,2.769464e+07
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,NaN,NaN,1.000000e+00,NaN,NaN,1.000000e+00,NaN,1.000000e+00,NaN,NaN,NaN,1.000000e+00,1.000000e+00,NaN,1.000000e+00,NaN
sf_tf05_credit_card_leakage,9.659000e-01,8.220000e-01,9.144000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.568000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.386000e-01,9.568000e-01,9.477000e-01,8.894000e-01,8.947000e-01,9.659000e-01,9.659000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,9.806000e-01,9.796000e-01,NaN,NaN,9.803000e-01,NaN,NaN,9.777000e-01,NaN,9.802000e-01,NaN,NaN,NaN,9.810000e-01,9.801000e-01,NaN,9.800000e-01,NaN


In [12]:
summary = {"stage1_fix": s1.reset_index().to_dict("records"), "stage1_wrong": wr.reset_index().to_dict("records"),
  "reasons_rejected_instr": rej.groupby(["arm", "kind"]).instr.agg(["sum", "count"]).reset_index().to_dict("records"),
  "judgements": jr.groupby(["arm", "kind", "useful"]).size().reset_index(name="n").to_dict("records"),
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "tf05_leak_cols_kept": tf05_kept,
  "fisher_p": fisher, "trust": fun.groupby(["arm", "shown"]).followed.agg(["sum", "count"]).reset_index().to_dict("records"),
  "trust_by_task": fun.groupby(["arm", "task"])[["shown", "followed"]].sum().reset_index().to_dict("records"),
  "score_mean": scores.to_dict("index"), "guide_tokens_per_run": gcost}
json.dump(summary, open(REPO / "quick_runs/results/P18/summary.json", "w"), indent=1, default=str)
jr.to_csv(REPO / "quick_runs/results/P18/judgements.csv", index=False); fun.to_csv(REPO / "quick_runs/results/P18/trust_funnel.csv", index=False)
"done" 

'done'